# ML BTC

## Imports

In [31]:
import os
import requests
from datetime import datetime

import kagglehub
import polars as pl

## Dataset loading

### Bitcoin historical data

In [40]:
path = kagglehub.dataset_download("adilshamim8/bitcoin-historical-data")
filename = "Bitcoin_history_data.csv"
filepath = os.path.join(path, filename)
df_btc = pl.read_csv(filepath)
df_btc

Date,Close,High,Low,Open,Volume
str,f64,f64,f64,f64,i64
"""2014-09-17""",457.334015,468.174011,452.421997,465.864014,21056800
"""2014-09-18""",424.440002,456.859985,413.104004,456.859985,34483200
"""2014-09-19""",394.79599,427.834991,384.532013,424.102997,37919700
"""2014-09-20""",408.903992,423.29599,389.882996,394.673004,36863600
"""2014-09-21""",398.821014,412.425995,393.181,408.084991,26580100
…,…,…,…,…,…
"""2026-02-16""",68843.15625,70067.234375,67301.585938,68782.398438,33618145426
"""2026-02-17""",67494.21875,69201.867188,66615.28125,68843.09375,34866936040
"""2026-02-18""",66425.320312,68434.429688,65845.898438,67488.023438,33094301643


### Fear and greed index

In [ ]:
url = "https://api.alternative.me/fng/?limit=0&format=json"
response = requests.get(url)
data = response.json()["data"]

rows = []
for row in data:
    timestamp = int(row["timestamp"])
    date = str(datetime.fromtimestamp(timestamp).date())
    value = row["value_classification"]
    rows.append({"Date": date, "Fear and greed": value})

df_fear_and_greed = pl.DataFrame(rows)
df_fear_and_greed

Date,Fear and greed
str,str
"""2026-03-02""","""Extreme Fear"""
"""2026-03-01""","""Extreme Fear"""
"""2026-02-28""","""Extreme Fear"""
"""2026-02-27""","""Extreme Fear"""
"""2026-02-26""","""Extreme Fear"""
…,…
"""2018-02-05""","""Extreme Fear"""
"""2018-02-04""","""Extreme Fear"""
"""2018-02-03""","""Fear"""


### Dataset merging

In [64]:
df = df_btc.join(df_fear_and_greed, on="Date", how="outer")
df

/tmp/ipykernel_33312/317962971.py:1: DeprecationWarning: use of `how='outer'` should be replaced with `how='full'`.
(Deprecated in version 0.20.29)
  df = df_btc.join(df_fear_and_greed, on="Date", how="outer")


Date,Close,High,Low,Open,Volume,Date_right,Fear and greed
str,f64,f64,f64,f64,i64,str,str
"""2014-09-17""",457.334015,468.174011,452.421997,465.864014,21056800,null,null
"""2014-09-18""",424.440002,456.859985,413.104004,456.859985,34483200,null,null
"""2014-09-19""",394.79599,427.834991,384.532013,424.102997,37919700,null,null
"""2014-09-20""",408.903992,423.29599,389.882996,394.673004,36863600,null,null
"""2014-09-21""",398.821014,412.425995,393.181,408.084991,26580100,null,null
…,…,…,…,…,…,…,…
null,null,null,null,null,null,"""2026-02-21""","""Extreme Fear"""
null,null,null,null,null,null,"""2026-02-23""","""Extreme Fear"""
null,null,null,null,null,null,"""2026-02-28""","""Extreme Fear"""


## Technical indicators calculation

These indicators are calculated before the train/test split because they are based only on the preceding N days of data.
Without this historical data, it would not be possible to compute these indicators.

### Weekly data

Some indicators are calculated on a weekly basis because this is more appropriate for long-term analysis.

In [ ]:
df_tmp = df.with_columns(pl.col("Date").str.strptime(pl.Datetime, format="%Y-%m-%d"))

df_tmp["week"] = df_tmp["Date"].dt.strftime("%Y-W%U")

df_tmp = df_tmp.filter(pl.col("Date").dt.weekday() == 7) # 7 = sunday
df_tmp = df.with_columns(pl.col("Date").dt.strptime(pl.Datetime, format="%Y-%w"))
df_weekly = df_tmp.select([
    "Date",
    "Close"
])
df_weekly

TypeError: DataFrame object does not support `Series` assignment by index

Use `DataFrame.with_columns`.

### Relative Strength Index (RSI)

This index measures both the strength and the speed of a price movement.
A value above 70 indicates that BTC is overbought, while a value below 30 indicates that BTC is oversold.
For the long term, the weekly RSI is the most appropriate indicator.

Date,Close_weekly
str,f64
"""2014-09-18""",424.440002
"""2014-09-25""",411.574005
"""2014-10-02""",375.071991
"""2014-10-09""",365.026001
"""2014-10-16""",382.556
…,…
"""2026-01-22""",89462.453125
"""2026-01-29""",84561.585938
"""2026-02-05""",62702.097656
